# TS76 Skeletonization

## Setup

In [1]:
from mcf2swc import *

from swctools import SWCModel, FrustaSet, PointSet, plot_model

import logging
logging.basicConfig(level=logging.INFO)

import os

print("✅ Libraries imported successfully!")

✅ Libraries imported successfully!


## Load Mesh and skeleton

In [ ]:
spine_idx = 76
mcf_qst = 1.3
mcf_mcst = 5
obj_name = f"TS{spine_idx}"
skeleton_name = f"TS{spine_idx}_qst{mcf_qst}_mcst{mcf_mcst}"
mm = MeshManager(mesh_path=f"../data/mesh/processed/{obj_name}_simplified.obj")
raw_skeleton = SkeletonGraph.from_txt(f"../data/mcf_skeletons/{skeleton_name}.polylines.txt")
raw_skeleton.prune_short_branches_inplace(min_length=10)
mm.visualize_mesh_3d(skel=raw_skeleton)

INFO:mcf2swc.mesh:Loaded mesh: 1509 vertices, 3022 faces


SkeletonGraph with 138 nodes and 137 edges


# Skeleton optimization

In [ ]:
from mcf2swc import (
    SkeletonGraph,
    MeshManager,
    SkeletonOptimizer,
    SkeletonOptimizerOptions,
)

do_skeleton_optimization = True

if do_skeleton_optimization:
    opts = SkeletonOptimizerOptions(
        max_iterations=10,
        step_size=1.0,
        smoothing_weight=0.1,
        verbose=True,
    )
    # Create optimizer and check for surface crossing
    optimizer = SkeletonOptimizer(raw_skeleton, mm.mesh, opts)
    has_crossing, num_outside, max_dist = optimizer.check_surface_crossing()
    print(f"Surface crossing: {has_crossing}")
    print(f"Points outside: {num_outside}/{raw_skeleton.total_points()}")
    print(f"Max distance: {max_dist:.4f}")
    # Run optimization
    print("\nOptimizing skeleton...")
    optimized_skeleton = optimizer.optimize()

    skeleton = optimized_skeleton
    all_skeletons = [raw_skeleton, optimized_skeleton]
else:
    skeleton = raw_skeleton
    all_skeletons = [raw_skeleton]


mm.visualize_mesh_3d(skel=all_skeletons, skel_color=["crimson", "blue"])


INFO:mcf2swc.skeleton_optimizer:No surface crossing detected - all nodes inside mesh
INFO:mcf2swc.skeleton_optimizer:No surface crossing detected - all nodes inside mesh
INFO:mcf2swc.skeleton_optimizer:Starting skeleton optimization...
INFO:mcf2swc.skeleton_optimizer:  Nodes: 138
INFO:mcf2swc.skeleton_optimizer:  Max iterations: 10
INFO:mcf2swc.skeleton_optimizer:  Step size: 1.0000
INFO:mcf2swc.skeleton_optimizer:  Smoothing weight: 0.1000


Surface crossing: False
Points outside: 0/138
Max distance: 0.0000

Optimizing skeleton...


INFO:mcf2swc.skeleton_optimizer:  Iteration 0: avg movement = 0.904214
INFO:mcf2swc.skeleton_optimizer:Optimization complete


# Fit morphology

In [6]:
spacing = 100

swc_out_dir = f"../data/swc/current/{skeleton_name}"

# check if directory exists, if not create it
if not os.path.exists(swc_out_dir):
    os.makedirs(swc_out_dir)

radius_strategy_list = [
    "equivalent_area",
    # "equivalent_perimeter",
    # "section_median",
    # "section_circle_fit",
    # "nearest_surface",
]
for radius_strategy in radius_strategy_list:
    print(f"Computing skeleton for radius_strategy={radius_strategy} ...", end="")
    morph = fit_morphology(
        mm.mesh, 
        skeleton, 
        options=FitOptions(
        spacing=spacing,
        radius_strategy=radius_strategy,
        snap_polylines_to_mesh=False)
)
    # write swc to file
    morph.to_swc_file(f"{swc_out_dir}/TS{spine_idx}_s{spacing}_{radius_strategy}.swc")
    morph.print_attributes(node_info=False, edge_info=False)
    print("DONE")

INFO:mcf2swc.graph_fitting:Tracing start: mesh[V=1509,F=3022], skeleton[nodes=138,edges=137], spacing=100, radius_strategy=equivalent_area
INFO:mcf2swc.graph_fitting:Edge group 0: input_pts=26 -> samples=4
INFO:mcf2swc.graph_fitting:Edge group 1: input_pts=25 -> samples=4
INFO:mcf2swc.graph_fitting:Edge group 2: input_pts=37 -> samples=5


Computing skeleton for radius_strategy=equivalent_area ...

INFO:mcf2swc.graph_fitting:Edge group 3: input_pts=17 -> samples=3
INFO:mcf2swc.graph_fitting:Edge group 4: input_pts=37 -> samples=5
INFO:mcf2swc.graph_fitting:Tracing done: nodes=17, edges=16, samples=21, section=21, fallback=0


MorphologyGraph: nodes=17, edges=16, components=1, cycles=0, branch_points=2, leaves=4, self_loops=0, density=0.1176
DONE


# Plotting

In [7]:
# plot using swctools
make_html = False

for radius_strategy in radius_strategy_list:
    swc_filepath = f"{swc_out_dir}/TS{spine_idx}_s{spacing}_{radius_strategy}.swc"
    model = SWCModel.from_swc_file(swc_filepath)
    model.print_attributes(node_info=False, edge_info=False)
    frusta = FrustaSet.from_swc_model(model)
    title = f"TS{spine_idx}_s{spacing}_{radius_strategy}"
    fig = plot_model(swc_model=model, frusta=frusta, slider=True, title=title)
    fig.show()
    if make_html:
        fig.write_html(f"../plotly/TS{spine_idx}_s{spacing}_{radius_strategy}.html")


INFO:swctools.io:parse_swc start strict=True validate_reconnections=True float_tol=1e-09
INFO:swctools.io:parse_swc done records=17 reconnections=0 header=4
INFO:swctools.model:SWCModel.from_parse_result records=17 reconnections=0 header=4
INFO:swctools.model:SWCModel.from_swc_file built nodes=17 edges=16 strict=True validate_reconnections=True
INFO:swctools.geometry:batch_frusta count=16 sides=16 end_caps=False verts=512 faces=512
INFO:swctools.geometry:FrustaSet.from_swc_model edges=16 sides=16 end_caps=False
INFO:swctools.geometry:batch_frusta count=16 sides=16 end_caps=False verts=512 faces=512
INFO:swctools.geometry:FrustaSet.scaled radius_scale=0.0
INFO:swctools.geometry:batch_frusta count=16 sides=16 end_caps=False verts=512 faces=512
INFO:swctools.geometry:FrustaSet.scaled radius_scale=0.05
INFO:swctools.geometry:batch_frusta count=16 sides=16 end_caps=False verts=512 faces=512
INFO:swctools.geometry:FrustaSet.scaled radius_scale=0.1
INFO:swctools.geometry:batch_frusta count=16

SWCModel: nodes=17, edges=16, components=1, cycles=0, branch_points=2, roots=1, leaves=4, self_loops=0, density=0.1176


INFO:swctools.viz:plot_model slider=True segments=16 radius_scale_range=[0.0,1.0]
